In [ ]:
from src.model.MultiU_NetModel import MultiU_Net
from src.training.out_conv_training.SortingMethod import BEST_PERM
from src.model.MaskTransform import multi_class_post_process
from src.model.MaskVisualization import segmentation_visualizer


In [ ]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub

import torch

from src.benchmark.ModelBenchmark import evaluate_model_on_dataset
from src.dataset.DroneSegDataSet import MyDataset
from src.dataset.CheckDataset import check_dataset
from torchvision import transforms

# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]),
    # data_enforcement=True,
)

n_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 数据集大小400 随机获取5个样本的索引
import random
sample_indices = random.sample(range(len(dataset)), 5)

from src.model.MultiU_NetModel import MultiU_Net
model = MultiU_Net(
    in_channel=22,
    depth=[3] * 5,
    depthwise_separable=False,
    combine_method='out_layer',
).to(device)

import os
para_dir = 'resources/para'
para_list = [
    os.path.join(para_dir, 'unet_branch0.pth'),
    os.path.join(para_dir, 'unet_branch1.pth'),
    os.path.join(para_dir, 'unet_branch2.pth'),
    os.path.join(para_dir, 'unet_branch3.pth'),
    os.path.join(para_dir, 'unet_branch4.pth'),
]
out_para = os.path.join(para_dir, 'out_conv.pth')
model.read_param(para_list, out_para)

for idx in sample_indices:
    feat_img, label, img = dataset.__getitem__(idx)

    feat_img = feat_img.unsqueeze(0).to(device) # 添加批次维度
    with torch.no_grad():
        output = model(feat_img)

    fixed_output = multi_class_post_process(output, BEST_PERM)

    # 可视化
    visualized = segmentation_visualizer(
        image=img, # 转换为 HWC 格式
        mask=fixed_output.squeeze(0).cpu(), # 去除批次维度并转换为 numpy 数组
    )

    # 展示可视化的图片、原图、标签
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0)) # 转换为 HWC 格式
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(label.cpu(), cmap='jet', vmin=0, vmax=n_classes-1)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')

    axes[2].imshow(visualized)
    axes[2].set_title('Model Prediction')
    axes[2].axis('off')
